# Notebook 04: Speech Recognition (ASR)

**Time:** 25 minutes  
**Prerequisites:** Notebook 03 complete  
**Goal:** Transcribe audio using Whisper and faster-whisper, compare speed and accuracy

This notebook will:
1. Transcribe audio with OpenAI Whisper (baseline)
2. Use faster-whisper for 4x speedup with CTranslate2
3. Compare model sizes and their speed/accuracy trade-offs
4. Discuss real-world ASR for pretraining data (podcasts, lectures, YouTube)

> **Why this matters:** Audio is a massive untapped data source for LLM pretraining. Podcasts, lectures, and YouTube videos contain billions of tokens of high-quality spoken content. ASR converts this audio to text. Whisper v3 turbo (2025) processes audio 216x faster than real-time.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

import src.audio_utils
importlib.reload(src.audio_utils)
from src.audio_utils import (
    transcribe_with_faster_whisper,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 04")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 04


---

## Part 1: OpenAI Whisper (Baseline)

Whisper is OpenAI's open-source ASR model. It supports 100+ languages and multiple model sizes:

| Model | Params | Speed | Best For |
|-------|--------|-------|----------|
| tiny | 39M | Fastest | Quick testing |
| base | 74M | Fast | Good default |
| small | 244M | Medium | Better accuracy |
| medium | 769M | Slow | High accuracy |
| large-v3 | 1.5B | Slowest | Best accuracy |
| turbo | 809M | 6x faster than large | Best speed/accuracy (2025) |

In [2]:
print("=" * 65)
print("Experiment 1: OpenAI Whisper Transcription")
print("=" * 65)
print()

# Find test audio
test_audio = os.path.join(parent_dir, 'test_data', 'audio', 'sample-1.mp3')

if not os.path.exists(test_audio):
    # Try Class3 test data
    class3_audio = os.path.join(parent_dir, '..', 'MLE_in_Gen_AI-Course', 'Class3', 'test_data', 'audio', 'sample-1.mp3')
    if os.path.exists(class3_audio):
        test_audio = class3_audio
    else:
        print("No test audio found.")
        print("Place an audio file at: test_data/audio/sample-1.mp3")
        test_audio = None

if test_audio:
    try:
        from src.audio_utils import transcribe_with_whisper
        whisper_result = transcribe_with_whisper(test_audio, model_size="base")
    except ImportError as e:
        print(f"Whisper not installed: {e}")
        print("Install with: pip install openai-whisper")
        print("Continuing with faster-whisper in Part 2...")

Experiment 1: OpenAI Whisper Transcription

WHISPER ASR (model: base)
Audio: c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\sample-1.mp3


c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Language: en
  Segments: 3
  Transcribed in 14.9s
  Text: I was able to, you know to live by. That's the......


---

## Part 2: faster-whisper (4x Speedup)

**faster-whisper** uses CTranslate2 to run Whisper models 4x faster with lower memory. It's the recommended backend for production ASR.

Key features:
- Supports all Whisper model sizes including **turbo**
- Batched inference for processing multiple files
- INT8 quantization for CPU efficiency
- Word-level timestamps

In [3]:
print("=" * 65)
print("Experiment 2: faster-whisper Transcription")
print("=" * 65)
print()

if test_audio:
    fw_result = transcribe_with_faster_whisper(
        test_audio,
        model_size="base",
        device="cpu",
        compute_type="int8"
    )
    
    print("\nTimestamped segments:")
    for seg in fw_result['segments']:
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")
else:
    print("No test audio available.")

Experiment 2: faster-whisper Transcription

FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\sample-1.mp3
  Language: en (prob: 0.621)
  Segments: 3
  Transcribed in 3.1s
  Text: The way he pays for that, they didn't, you know, that he's a people's secretary son. Cren the Benzo instead of court and a musical or just not to worry about your interests and just turning child, tha...

Timestamped segments:
  [0.0s - 4.0s] The way he pays for that, they didn't, you know, that he's a people's secretary son.
  [4.0s - 8.0s] Cren the Benzo instead of court and a musical or just not to worry about your interests and
  [8.0s - 10.0s] just turning child, that's the...


In [4]:
print("=" * 65)
print("Experiment 3: Model Size Comparison")
print("=" * 65)
print()

if test_audio:
    model_sizes = ["tiny", "base", "small"]
    results = {}
    
    for size in model_sizes:
        print(f"\n--- Testing model: {size} ---")
        result = transcribe_with_faster_whisper(
            test_audio, model_size=size, device="cpu", compute_type="int8"
        )
        results[size] = result
    
    print("\n\n--- Speed Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['elapsed_seconds']:.2f}s")
    
    print("\n--- Transcription Comparison ---")
    for size, res in results.items():
        print(f"  {size:>8}: {res['text'][:100]}...")
else:
    print("No test audio available.")

Experiment 3: Model Size Comparison


--- Testing model: tiny ---
FASTER-WHISPER ASR (model: tiny, cpu/int8)
Audio: c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\sample-1.mp3


tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wesle\.cache\huggingface\hub\models--Systran--faster-whisper-tiny. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

  Language: en (prob: 0.809)
  Segments: 4
  Transcribed in 11.2s
  Text: the way he pays a bat, they didn't, you know, that he's some people's sector son. Kren, the benso, instead of court and a musical or just not story of which you're interested in just turning child, th...

--- Testing model: base ---
FASTER-WHISPER ASR (model: base, cpu/int8)
Audio: c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\sample-1.mp3
  Language: en (prob: 0.621)
  Segments: 3
  Transcribed in 2.7s
  Text: The way he pays for that, they didn't, you know, that he's a people's secretary son. Cren the Benzo instead of court and a musical or just not to worry about your interests and just turning child, tha...

--- Testing model: small ---
FASTER-WHISPER ASR (model: small, cpu/int8)
Audio: c:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\sample-1.mp3
  Language: en (prob: 0.857)
  Segments: 5
  Transcribed in 7.6s
  Text: The way he pays for that, they didn't,

In [6]:
# TODO 1: Analyze ASR for pretraining data
#
# Use the LLM to discuss how ASR fits into a pretraining pipeline.

print("=" * 65)
print("TODO 1: ASR in Pretraining Pipelines")
print("=" * 65)
print()

start = time.time()
response = client.generate(
    prompt="""Explain how Automatic Speech Recognition (ASR) is used to build LLM pretraining datasets.

Cover:
1. What audio sources are typically transcribed (podcasts, lectures, YouTube)?
2. What quality issues arise from ASR transcriptions vs written text?
3. How do modern models like Whisper v3 turbo (2025) compare to earlier ASR?
4. What post-processing is needed before ASR text goes into a training dataset?

Be practical and give specific examples.""",
    system="You are an expert in building LLM pretraining data pipelines.",
    max_tokens=500,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo1_reflection = """
[YOUR REFLECTION HERE]

- How does ASR transcription quality compare to web-scraped text?
- I think ASR transcription quality can vary widely based on the model used, the audio quality, and the speaker's accent. While web-scraped text is often cleaner and more structured, ASR transcriptions can contain errors, misheard words, and lack punctuation. However, modern ASR models like Whisper v3 turbo have made significant improvements in accuracy, especially for clear audio. Post-processing steps such as punctuation restoration and error correction can help improve the quality of ASR text before it is used in training datasets.
- Which model size would you choose for transcribing 1000 hours of podcasts?
- I will chose the "base" model size for transcribing 1000 hours of podcasts. The "tiny" model may be too inaccurate for large-scale transcription, while the "small" model may be unnecessarily slow and resource-intensive. The "base" model offers a good balance of speed and accuracy, making it suitable for processing a large volume of audio data efficiently while maintaining reasonable transcription quality.
- What are the ethical considerations of transcribing public audio content?
- Ethical considerations include respecting privacy and consent, especially if the audio contains personal or sensitive information. Even if the content is publicly available, it may not be ethical to transcribe and use it without permission from the speakers. There is also the risk of misrepresentation if the ASR transcription contains errors, which could lead to misinformation or harm to individuals. Additionally, there may be copyright issues when transcribing and using content from podcasts or YouTube videos, so it's important to ensure that the content is used in compliance with copyright laws and platform policies.
"""

print()
print(todo1_reflection)

TODO 1: ASR in Pretraining Pipelines

Response in 15.1s
Model: claude-sonnet-4-6
Tokens: 137 in, 500 out
Stop reason: max_tokens
# ASR for LLM Pretraining Datasets

## Why Bother With Audio at All?

The core motivation is simple: **spoken language contains content that never gets written down.**

Conversational explanations, expert reasoning-out-loud, informal technical discussions, storytelling cadence — these linguistic patterns exist almost exclusively in audio form. A model trained only on written text learns formal register heavily and misses the way humans actually explain things to each other. Audio transcription is one of the few ways to get that data at scale.

The secondary motivation is **volume**. Written text on the internet is increasingly synthetic or low-quality. High-quality human speech — a professor explaining thermodynamics, a doctor discussing a diagnosis, an engineer debugging live — represents genuine human knowledge production that hasn't been strip-mined yet.



In [12]:
# TODO 2: Transcribe your own audio (optional)
#
# Record a 15-30 second audio clip or find one online.
# Transcribe it and evaluate the quality.

print("=" * 65)
print("TODO 2: Custom Audio Transcription (Optional)")
print("=" * 65)
print()

my_audio = "C:\\Users\\wesle\\Desktop\\HW\\wesley_HW3\\Homework3-Submission\\test_data\\audio\\DevFest_2025_Recap.mp3"
my_result = transcribe_with_faster_whisper(my_audio, model_size="small")

todo2_reflection = """
[YOUR REFLECTION HERE]

- What audio did you transcribe?
- I transcribed a podcast episode titled "DevFest 2025 Recap," which is a summary of the key highlights and announcements from the DevFest 2025 conference.
- How accurate was the transcription?
- The accuracy of the transcription was quite good, especially for clear sections of the audio. However, there were some errors in transcribing technical terms and names, which is a common issue with ASR. Overall, the transcription captured the main content of the podcast, but it may require some manual correction for specific details.
- What would you change for better results (model size, preprocessing)?
- For better results, I might experiment with the "base" model size to see if it improves accuracy, especially for technical terms. Additionally, I could apply some preprocessing to enhance the audio quality, such as noise reduction or normalization, which might help the ASR model perform better. Post-processing the transcription with a tool that corrects common ASR errors or restores punctuation could also improve the readability and usefulness of the transcribed text.
"""

print(todo2_reflection)

TODO 2: Custom Audio Transcription (Optional)

FASTER-WHISPER ASR (model: small, cpu/int8)
Audio: C:\Users\wesle\Desktop\HW\wesley_HW3\Homework3-Submission\test_data\audio\DevFest_2025_Recap.mp3
  Language: en (prob: 0.969)
  Segments: 17
  Transcribed in 17.0s
  Text: Hi, everyone. Welcome to DevFest. We're in a very exciting climb. We have over 700 DevFest happening across the world. Those are events organized for the community and by the community. It's been a fa...

[YOUR REFLECTION HERE]

- What audio did you transcribe?
- I transcribed a podcast episode titled "DevFest 2025 Recap," which is a summary of the key highlights and announcements from the DevFest 2025 conference.
- How accurate was the transcription?
- The accuracy of the transcription was quite good, especially for clear sections of the audio. However, there were some errors in transcribing technical terms and names, which is a common issue with ASR. Overall, the transcription captured the main content of the podcast, 

---

## Summary & Reflection

In [13]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'

full_reflection = f"""
### Part 1 - ASR in Pretraining

{_todo1}

---

### Part 2 - Custom Transcription

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="04",
    section_title="Speech Recognition (ASR)",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     2
Total input tokens:  274
Total output tokens: 1,000
Total cost:          $0.0158

Last 2 calls:
  1. [22:19:58] sonnet -- 137in/500out -- $0.0079
  2. [22:23:33] sonnet -- 137in/500out -- $0.0079


## Notebook 04 Complete!

**What you accomplished:**
- Transcribed audio with OpenAI Whisper and faster-whisper
- Compared model sizes for speed vs accuracy
- Explored how ASR fits into pretraining pipelines

**Key concepts:**
- faster-whisper provides 4x speedup with CTranslate2 backend
- Whisper v3 turbo (2025) offers best speed/accuracy trade-off
- ASR text needs post-processing before use in pretraining

**Next:** Open **Notebook 05: Data Cleaning Pipeline**